# Trust-Aware Federated Multimodal Graph Recommendation System

This notebook demonstrates the complete implementation of our BTP project.

## Architecture Overview

### 🔵 Layer 1: Client Side (User Devices)
- Private data storage (reviews, images, interactions)
- Local training and modality encoders

### 🟣 Layer 2: Federated Server
- Trust-weighted aggregation
- Model update collection and distribution

### 🟢 Layer 3: Graph + Recommendation Engine
- User-Item graph construction
- GNN-based recommendations
- Trust score computation

## 1. Setup and Imports

In [ ]:
# Install required packages if not already installed
# !pip install torch torch-geometric flwr numpy pandas scikit-learn matplotlib seaborn transformers pillow

import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Import our modules
from data.dataset_preparation import DatasetPreparation
from models.encoders.multimodal_encoders import RecommendationEncoder, TextEncoder, ImageEncoder
from models.gnn.graph_models import BipartiteGraphRecommender, GraphConstructor
from models.trust.trust_mechanism import TrustMechanism, TrustAwareAggregator
from utils.recommendation_system import RecommendationSystem, RecommendationAPI

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Dataset Preparation

In [ ]:
# Initialize dataset preparation
dataset_prep = DatasetPreparation()

# Create sample dataset
print("📊 Creating sample dataset...")
client_data, metadata = dataset_prep.prepare_full_dataset(
    num_users=100,
    num_items=50,
    num_interactions=500,
    num_clients=5
)

print(f"\n✅ Dataset created:")
print(f"   - Users: {metadata['num_users']}")
print(f"   - Items: {metadata['num_items']}")
print(f"   - Text feature dimension: {metadata['text_feature_dim']}")
print(f"   - Clients: {len(client_data)}")

# Display data distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
client_sizes = [len(client['user_ids']) for client in client_data]
plt.bar(range(len(client_sizes)), client_sizes)
plt.title('Users per Client')
plt.xlabel('Client ID')
plt.ylabel('Number of Users')

plt.subplot(1, 2, 2)
interaction_sizes = [len(client['ratings']) for client in client_data]
plt.bar(range(len(interaction_sizes)), interaction_sizes)
plt.title('Interactions per Client')
plt.xlabel('Client ID')
plt.ylabel('Number of Interactions')

plt.tight_layout()
plt.show()

## 3. Multi-Modal Encoders

In [ ]:
# Initialize encoders
print("🔧 Initializing multi-modal encoders...")

text_encoder = TextEncoder(
    input_dim=metadata['text_feature_dim'],
    output_dim=128,
    use_transformer=False  # Use TF-IDF for simplicity
)

image_encoder = ImageEncoder(output_dim=128)

multimodal_encoder = RecommendationEncoder(
    num_users=metadata['num_users'],
    num_items=metadata['num_items'],
    text_input_dim=metadata['text_feature_dim'],
    multimodal_dim=256
)

print(f"✅ Encoders initialized:")
print(f"   - Text encoder: {sum(p.numel() for p in text_encoder.parameters())} parameters")
print(f"   - Image encoder: {sum(p.numel() for p in image_encoder.parameters())} parameters")
print(f"   - Multimodal encoder: {sum(p.numel() for p in multimodal_encoder.parameters())} parameters")

# Test encoders with dummy data
batch_size = 16
dummy_text_features = torch.randn(batch_size, metadata['text_feature_dim'])
dummy_images = torch.randn(batch_size, 3, 224, 224)
dummy_user_ids = torch.randint(0, metadata['num_users'], (batch_size,))
dummy_item_ids = torch.randint(0, metadata['num_items'], (batch_size,))

with torch.no_grad():
    final_emb, embeddings_dict = multimodal_encoder(
        dummy_user_ids, dummy_item_ids, dummy_text_features, dummy_images
    )

print(f"\n🧪 Test results:")
print(f"   - Final embedding shape: {final_emb.shape}")
print(f"   - User embedding shape: {embeddings_dict['user_emb'].shape}")
print(f"   - Item embedding shape: {embeddings_dict['item_emb'].shape}")
print(f"   - Text embedding shape: {embeddings_dict['text_emb'].shape}")
print(f"   - Image embedding shape: {embeddings_dict['image_emb'].shape}")

## 4. Graph Neural Network

In [ ]:
# Initialize GNN
print("🕸️ Initializing Graph Neural Network...")

gnn = BipartiteGraphRecommender(
    num_users=metadata['num_users'],
    num_items=metadata['num_items'],
    user_dim=64,
    item_dim=64,
    hidden_dim=128,
    output_dim=64
)

print(f"✅ GNN initialized: {sum(p.numel() for p in gnn.parameters())} parameters")

# Test GNN with dummy data
with torch.no_grad():
    predictions, user_emb, item_emb = gnn(
        dummy_user_ids, dummy_item_ids, final_emb
    )

print(f"\n🧪 Test results:")
print(f"   - Predictions shape: {predictions.shape}")
print(f"   - Prediction range: [{predictions.min():.3f}, {predictions.max():.3f}]")
print(f"   - User embeddings shape: {user_emb.shape}")
print(f"   - Item embeddings shape: {item_emb.shape}")

# Visualize prediction distribution
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(predictions.detach().numpy(), bins=20, alpha=0.7)
plt.title('Prediction Distribution')
plt.xlabel('Predicted Score')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.scatter(user_emb[:, 0].detach().numpy(), user_emb[:, 1].detach().numpy(), alpha=0.6, label='Users')
plt.scatter(item_emb[:, 0].detach().numpy(), item_emb[:, 1].detach().numpy(), alpha=0.6, label='Items')
plt.title('User-Item Embeddings (First 2 dims)')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.legend()

plt.tight_layout()
plt.show()

## 5. Trust Mechanism

In [ ]:
# Initialize trust mechanism
print("🛡️ Initializing Trust Mechanism...")

trust_mechanism = TrustMechanism(
    history_length=10,
    trust_decay=0.9,
    consistency_weight=0.3,
    performance_weight=0.4,
    similarity_weight=0.3
)

print("✅ Trust mechanism initialized")

# Simulate multiple rounds of federated learning
print("\n🔄 Simulating trust score evolution...")

simulated_clients = ['client_0', 'client_1', 'client_2', 'client_3', 'client_4']
trust_evolution = {client: [] for client in simulated_clients}

for round_num in range(10):
    # Simulate client metadata
    client_metadata = []
    for i, client_id in enumerate(simulated_clients):
        # Simulate different client behaviors
        if client_id == 'client_0':  # Good client
            loss = 0.3 + np.random.normal(0, 0.05)
        elif client_id == 'client_1':  # Average client
            loss = 0.5 + np.random.normal(0, 0.1)
        elif client_id == 'client_2':  # Poor client
            loss = 0.8 + np.random.normal(0, 0.15)
        else:  # Random client
            loss = np.random.uniform(0.2, 0.9)
        
        client_metadata.append({
            'client_id': client_id,
            'loss': max(0.1, loss),  # Ensure positive loss
            'num_examples': np.random.randint(50, 200)
        })
    
    # Calculate trust scores
    trust_scores = trust_mechanism.calculate_trust_scores(client_metadata, round_num)
    
    # Store trust evolution
    for i, client_id in enumerate(simulated_clients):
        trust_evolution[client_id].append(trust_scores[i])

# Plot trust evolution
plt.figure(figsize=(12, 6))
for client_id in simulated_clients:
    plt.plot(trust_evolution[client_id], marker='o', label=client_id)

plt.title('Trust Score Evolution Over Rounds')
plt.xlabel('Federated Learning Round')
plt.ylabel('Trust Score')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Get final trust statistics
stats = trust_mechanism.get_trust_statistics()
print(f"\n📊 Final Trust Statistics:")
print(f"   - Average reputation: {stats['avg_reputation']:.3f}")
print(f"   - Reputation std: {stats['reputation_std']:.3f}")

for client_id in simulated_clients:
    reputation = trust_mechanism.get_client_reputation(client_id)
    print(f"   - {client_id} reputation: {reputation:.3f}")

## 6. Complete Recommendation System

In [ ]:
# Initialize complete recommendation system
print("🎯 Initializing Complete Recommendation System...")

recommendation_system = RecommendationSystem(
    encoder=multimodal_encoder,
    gnn=gnn,
    trust_mechanism=trust_mechanism,
    num_users=metadata['num_users'],
    num_items=metadata['num_items']
)

# Set item metadata
item_metadata = {
    i: {
        'name': f'Product {i}',
        'category': f'Category {i % 5}',
        'price': f'${10.0 + i * 2.5:.2f}',
        'description': f'This is product {i} from category {i % 5}'
    } for i in range(metadata['num_items'])
}
recommendation_system.set_item_metadata(item_metadata)

# Add some user history
print("📝 Adding user interaction history...")
for user_id in range(10):
    for item_id in range(5):
        rating = np.random.uniform(1, 5)
        recommendation_system.update_user_history(user_id, item_id, rating)

print("✅ Recommendation system initialized")

# Test recommendations
print("\n🔍 Testing recommendations...")
test_user = 0
recommendations = recommendation_system.recommend_items_for_user(
    test_user, top_k=5, exclude_seen=True
)

print(f"\nTop 5 recommendations for User {test_user}:")
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. Item {rec['item_id']} ({rec['metadata']['name']})")
    print(f"   - Score: {rec['score']:.3f}")
    print(f"   - Predicted Rating: {rec['rating_prediction']:.1f}/5.0")
    print(f"   - Trust Score: {rec['trust_score']:.3f}")
    print(f"   - Reason: {rec['recommendation_reason']}")
    print()

## 7. Trust-Aware Recommendations

In [ ]:
# Compare regular vs trust-aware recommendations
print("🔍 Comparing Regular vs Trust-Aware Recommendations...")

test_user = 1
trust_threshold = 0.6

# Regular recommendations
regular_recs = recommendation_system.recommend_items_for_user(
    test_user, top_k=10, exclude_seen=True
)

# Trust-aware recommendations
trust_aware_recs = recommendation_system.get_trust_aware_recommendations(
    test_user, top_k=10, trust_threshold=trust_threshold
)

print(f"\nRegular Recommendations for User {test_user}:")
for i, rec in enumerate(regular_recs[:5], 1):
    trust_level = "High" if rec['trust_score'] > 0.7 else "Medium" if rec['trust_score'] > 0.4 else "Low"
    print(f"{i}. Item {rec['item_id']} - Score: {rec['score']:.3f}, Trust: {trust_level} ({rec['trust_score']:.3f})")

print(f"\nTrust-Aware Recommendations (threshold={trust_threshold}):")
for i, rec in enumerate(trust_aware_recs[:5], 1):
    trust_level = "High" if rec['trust_score'] > 0.7 else "Medium" if rec['trust_score'] > 0.4 else "Low"
    print(f"{i}. Item {rec['item_id']} - Score: {rec['score']:.3f}, Trust: {trust_level} ({rec['trust_score']:.3f})")

# Visualize trust score distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
regular_trust = [rec['trust_score'] for rec in regular_recs]
plt.hist(regular_trust, bins=10, alpha=0.7, label='Regular')
plt.axvline(trust_threshold, color='red', linestyle='--', label=f'Threshold ({trust_threshold})')
plt.title('Regular Recommendations - Trust Scores')
plt.xlabel('Trust Score')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 2, 2)
trust_aware_trust = [rec['trust_score'] for rec in trust_aware_recs]
plt.hist(trust_aware_trust, bins=10, alpha=0.7, color='green', label='Trust-Aware')
plt.axvline(trust_threshold, color='red', linestyle='--', label=f'Threshold ({trust_threshold})')
plt.title('Trust-Aware Recommendations - Trust Scores')
plt.xlabel('Trust Score')
plt.ylabel('Frequency')
plt.legend()

plt.tight_layout()
plt.show()

## 8. System Performance Evaluation

In [ ]:
# Evaluate recommendation system performance
print("📊 Evaluating System Performance...")

# Create test interactions
test_interactions = []
for user_id in range(20):
    item_id = np.random.randint(0, metadata['num_items'])
    rating = np.random.uniform(1, 5)
    test_interactions.append({
        'user_id': user_id,
        'item_id': item_id,
        'rating': rating
    })

# Evaluate recommendations
eval_results = recommendation_system.evaluate_recommendations(
    test_interactions, top_k=10
)

print(f"\n📈 Evaluation Results:")
print(f"   - Precision@10: {eval_results['precision_at_k']:.3f}")
print(f"   - Recall@10: {eval_results['recall_at_k']:.3f}")
print(f"   - NDCG@10: {eval_results['ndcg_at_k']:.3f}")
print(f"   - Evaluated interactions: {eval_results['num_evaluated']}")

# Test similar items functionality
print("\n🔗 Testing Similar Items...")
test_item = 0
similar_items = recommendation_system.get_similar_items(test_item, top_k=5)

print(f"Items similar to Item {test_item}:")
for i, item in enumerate(similar_items, 1):
    print(f"{i}. Item {item['item_id']} - Similarity: {item['similarity']:.3f}")
    if item['metadata']:
        print(f"   - {item['metadata']['name']} ({item['metadata']['category']})")

# Performance metrics visualization
metrics = ['Precision@10', 'Recall@10', 'NDCG@10']
values = [eval_results['precision_at_k'], eval_results['recall_at_k'], eval_results['ndcg_at_k']]

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
bars = plt.bar(metrics, values, color=['skyblue', 'lightgreen', 'salmon'])
plt.title('Recommendation Performance Metrics')
plt.ylabel('Score')
plt.ylim(0, 1)

# Add value labels on bars
for bar, value in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{value:.3f}', ha='center', va='bottom')

plt.subplot(1, 2, 2)
similarities = [item['similarity'] for item in similar_items]
item_ids = [item['item_id'] for item in similar_items]
plt.bar(range(len(similarities)), similarities)
plt.title('Item Similarities')
plt.xlabel('Similar Item ID')
plt.ylabel('Cosine Similarity')
plt.xticks(range(len(item_ids)), item_ids)

plt.tight_layout()
plt.show()

## 9. API Demonstration

In [ ]:
# Create API wrapper
print("🌐 Creating API Wrapper...")
api = RecommendationAPI(recommendation_system)

# Test API endpoints
print("\n📡 Testing API Endpoints...")

# Test recommendations endpoint
rec_response = api.get_recommendations(2, top_k=3, trust_aware=True)
print(f"\nRecommendations API Response:")
print(f"   - Success: {rec_response['success']}")
print(f"   - User ID: {rec_response['user_id']}")
print(f"   - Number of recommendations: {rec_response['num_recommendations']}")

# Test similar items endpoint
similar_response = api.get_similar_items(5, top_k=3)
print(f"\nSimilar Items API Response:")
print(f"   - Success: {similar_response['success']}")
print(f"   - Item ID: {similar_response['item_id']}")
print(f"   - Number of similar items: {similar_response['num_similar_items']}")

# Test interaction update endpoint
interaction_response = api.update_user_interaction(
    user_id=3, item_id=10, rating=4.5, review_text="Great product!"
)
print(f"\nInteraction Update API Response:")
print(f"   - Success: {interaction_response['success']}")
print(f"   - Message: {interaction_response.get('message', 'N/A')}")

print("\n✅ All API endpoints working correctly!")

## 10. Summary and Key Insights

In [ ]:
print("🎯 PROJECT SUMMARY")
print("=" * 50)

print("\n✅ Successfully Implemented:")
print("   1. Multi-modal encoders (text + image)")
print("   2. Graph Neural Network for recommendations")
print("   3. Trust mechanism with multiple scoring components")
print("   4. Trust-weighted federated aggregation")
print("   5. Complete recommendation system")
print("   6. RESTful API interface")

print("\n🔬 Key Components:")
print(f"   - Dataset: {metadata['num_users']} users, {metadata['num_items']} items")
print(f"   - Text encoder: {sum(p.numel() for p in text_encoder.parameters())} parameters")
print(f"   - Image encoder: {sum(p.numel() for p in image_encoder.parameters())} parameters")
print(f"   - GNN model: {sum(p.numel() for p in gnn.parameters())} parameters")
print(f"   - Trust mechanism: {len(trust_mechanism.client_reputations)} tracked clients")

print("\n📊 Performance Metrics:")
print(f"   - Precision@10: {eval_results['precision_at_k']:.3f}")
print(f"   - Recall@10: {eval_results['recall_at_k']:.3f}")
print(f"   - NDCG@10: {eval_results['ndcg_at_k']:.3f}")

print("\n🛡️ Trust Mechanism Features:")
print("   - Consistency scoring based on loss variance")
print("   - Performance scoring based on relative improvement")
print("   - Similarity scoring for update consistency")
print("   - Temporal decay for reputation management")
print("   - Trust-weighted aggregation strategies")

print("\n🚀 Next Steps for Production:")
print("   1. Real dataset integration (Amazon Reviews)")
print("   2. Advanced transformer-based text encoders")
print("   3. More sophisticated GNN architectures")
print("   4. Real-world federated deployment")
print("   5. Advanced trust mechanisms with adversarial robustness")

print("\n🎓 BTP Project Completed Successfully!")
print("   This implementation demonstrates a complete trust-aware federated")
print("   multimodal graph recommendation system suitable for BTP level.")